# LUCID-Lite — Notebook Visualizer

Access video frames and instance information from a running LUCID-Lite window.

**Prereqs:**
- A live `window` from `main.main([...])` (see §0 below)
- `matplotlib`, `pandas`, `numpy` (already in the venv except matplotlib/pandas — install if missing)

**Critical:** keep the `window` variable in scope. If it's GC'd, the panels' `DecodeWorker`s die and pending decode tasks crash with `RuntimeError: Signal source has been deleted`.

## 0. Launch LUCID-Lite from this notebook

Run this once. The window will open and stay live as long as `window` is held.

In [ ]:
%gui qt
import sys
from pathlib import Path

GUI_SRC = str(Path.cwd() / "gui_source")
if GUI_SRC not in sys.path:
    sys.path.insert(0, GUI_SRC)

import main

# No args -> folder picker dialog. To pass a folder:
#   app, window = main.main(["main.py", "/path/to/session"])
app, window = main.main([])
session = window.session
print("cameras:", session.camera_names())
print("frames:", session.min_frame, "->", session.max_frame)
print("tracks:", session.tracks)
print("identities:", [(i.id, i.name) for i in session.identities])

## Map of accessible state

```
window.session                          # Session (QObject)
window._video_panels                    # dict[cam, VideoPanelWidget]
window._video_panels[cam]._decoder      # OnDemandVideoDecoder (frames)
window._current_frame                   # current playhead

session.cameras                         # list[Camera] (calibration + size)
session.has_calibration                 # bool — False = synthesized stubs
session.camera_names()                  # list[str]
session.skeleton.nodes / .edges         # list[str] / list[(i,j)]
session.tracks                          # list[str]   track_idx -> name
session.identities                      # list[Identity]  (.id, .name, .color)
session.color_mode                      # "track" | "identity"
session.video_paths                     # dict[cam, Path]
session.frame_indices                   # sorted list of populated frames
session.min_frame, session.max_frame
session.frame_group(idx)                # FrameGroup | None
session.instance_groups_for(idx)        # list[InstanceGroup] (cross-cam)
session.get_identity_id_for_track(frame, cam, track_idx)
session.get_identity(identity_id)

FrameGroup.instances[cam]               # list[Instance]
Instance.points                         # list[(x,y) | None]  one per skeleton node
Instance.track_idx                      # int | None -> session.tracks[track_idx]
Instance.type                           # "user" | "predicted"
Instance.score                          # float

Camera.name / .matrix (K) / .dist / .rvec / .tvec / .size
Identity.id / .name / .color
```

**Gotchas**
- `Instance.points[i]` can be `None` — always guard.
- `Instance.track_idx` indexes `session.tracks`. `Identity.id` is an auto-counter. They are not interchangeable.
- Track colors are **derived** (`TRACK_COLORS[track_idx % 20]`), not stored. Identity colors **are** stored on each `Identity`.
- `session.has_calibration=False` is silent — `session.cameras` is still populated with identity-K stubs.

## 1. Get a frame as a NumPy array

`OnDemandVideoDecoder.get_frame(i)` is thread-safe and synchronous — seek-and-decode (or LRU cache hit).

In [ ]:
import numpy as np
from PySide6.QtGui import QImage

def qimage_to_numpy(qimg: QImage) -> np.ndarray:
    """QImage (any format) -> H×W×3 uint8 RGB array."""
    img = qimg.convertToFormat(QImage.Format_RGB888)
    w, h = img.width(), img.height()
    ptr = img.constBits()
    arr = np.frombuffer(ptr, dtype=np.uint8, count=img.sizeInBytes())
    arr = arr.reshape(h, img.bytesPerLine())[:, : w * 3]
    return arr.reshape(h, w, 3).copy()

cam = session.camera_names()[0]
decoder = window._video_panels[cam]._decoder
qimg = decoder.get_frame(window._current_frame)
frame = qimage_to_numpy(qimg)
print("shape:", frame.shape, " fps:", decoder.fps, " n_frames:", decoder.n_frames)

## 2. Plot a frame with the pose overlay

In [ ]:
import matplotlib.pyplot as plt

frame_idx = window._current_frame
cam = session.camera_names()[0]

img = qimage_to_numpy(window._video_panels[cam]._decoder.get_frame(frame_idx))
fg = session.frame_group(frame_idx)
skel = session.skeleton

fig, ax = plt.subplots(figsize=(10, 6))
ax.imshow(img)
ax.set_title(f"{cam}  frame {frame_idx}")

for inst in (fg.instances.get(cam, []) if fg else []):
    pts = inst.points
    # edges
    for a, b in skel.edges:
        if pts[a] is not None and pts[b] is not None:
            (xa, ya), (xb, yb) = pts[a], pts[b]
            ax.plot([xa, xb], [ya, yb], "-", linewidth=1.5)
    # nodes
    xs = [p[0] for p in pts if p is not None]
    ys = [p[1] for p in pts if p is not None]
    ax.scatter(xs, ys, s=18)
    # label at node 0
    track_name = (session.tracks[inst.track_idx]
                  if inst.track_idx is not None else "?")
    if pts[0] is not None:
        ax.annotate(track_name, pts[0], color="white", fontsize=9,
                    xytext=(5, -5), textcoords="offset points")

ax.axis("off")
plt.show()

## 3. Dump all instances at the current frame to a DataFrame

In [ ]:
import pandas as pd

fg = session.frame_group(window._current_frame)
rows = []
for cam, insts in (fg.instances if fg else {}).items():
    for inst in insts:
        rows.append({
            "cam": cam,
            "track_idx": inst.track_idx,
            "track": (session.tracks[inst.track_idx]
                      if inst.track_idx is not None else None),
            "identity_id": session.get_identity_id_for_track(
                window._current_frame, cam, inst.track_idx),
            "type": inst.type,
            "score": inst.score,
            "n_visible": sum(p is not None for p in inst.points),
        })
df = pd.DataFrame(rows)
df

## 4. All points for one (cam, track) over time

Returns a `(T, N_nodes, 2)` float array with `NaN` where a node wasn't visible.

In [ ]:
import numpy as np

def track_points(session, cam, track_idx):
    """Return (frames, xy) where xy is (T, N_nodes, 2) with NaN for missing."""
    N = len(session.skeleton.nodes)
    out, frames = [], []
    for f in session.frame_indices:
        fg = session.frame_group(f)
        if fg is None:
            continue
        for inst in fg.instances.get(cam, []):
            if inst.track_idx != track_idx:
                continue
            row = np.full((N, 2), np.nan, dtype=float)
            for i, p in enumerate(inst.points):
                if p is not None:
                    row[i] = p
            out.append(row)
            frames.append(f)
            break
    return np.asarray(frames), np.asarray(out)

cam = session.camera_names()[0]
frames, xy = track_points(session, cam, track_idx=0)
print("xy.shape =", xy.shape, "  first frames:", frames[:5])

### Quick trajectory plot for a single node

In [ ]:
import matplotlib.pyplot as plt

node_idx = 0  # change me
node_name = session.skeleton.nodes[node_idx]

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(frames, xy[:, node_idx, 0]); axes[0].set_ylabel(f"{node_name}.x")
axes[1].plot(frames, xy[:, node_idx, 1]); axes[1].set_ylabel(f"{node_name}.y")
axes[1].set_xlabel("frame")
axes[0].set_title(f"{cam} · track_idx=0 · node={node_name}")
plt.tight_layout(); plt.show()

## 5. Cross-camera grouping (`InstanceGroup`)

`InstanceGroup.instances` is singular per camera (`dict[cam, Instance]`).

In [ ]:
groups = session.instance_groups_for(window._current_frame)
for g in groups:
    print(f"group {g.id}  identity_id={g.identity_id}  cams={list(g.instances)}")
    for cam, inst in g.instances.items():
        n_vis = sum(p is not None for p in inst.points)
        print(f"  {cam}: track_idx={inst.track_idx}  n_visible={n_vis}")

## 6. Drive the UI from the notebook

All of these emit Qt signals; the GUI repaints automatically.

In [ ]:
# Seek every panel + timeline + assignment dock:
window.set_current_frame(120)

# Flip color mode:
# session.set_color_mode("identity")  # or "track"

# Assign per-frame identity for a (cam, track):
# session.set_frame_identity(120, session.camera_names()[0], track_idx=0, identity_id=2)

# Or globally:
# session.set_global_track_identity(session.camera_names()[0], track_idx=0, identity_id=2)

## 7. Side-by-side: all cameras at the same frame

In [ ]:
import math
import matplotlib.pyplot as plt

def show_all_cams(frame_idx: int, with_overlay: bool = True):
    cams = session.camera_names()
    n = len(cams)
    cols = math.ceil(math.sqrt(n))
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows), squeeze=False)
    fg = session.frame_group(frame_idx)
    skel = session.skeleton

    for k, cam in enumerate(cams):
        ax = axes[k // cols][k % cols]
        dec = window._video_panels[cam]._decoder
        qimg = dec.get_frame(frame_idx) if dec is not None else None
        if qimg is not None:
            ax.imshow(qimage_to_numpy(qimg))
        ax.set_title(f"{cam}  f={frame_idx}")
        ax.axis("off")
        if not with_overlay or fg is None:
            continue
        for inst in fg.instances.get(cam, []):
            pts = inst.points
            for a, b in skel.edges:
                if pts[a] is not None and pts[b] is not None:
                    (xa, ya), (xb, yb) = pts[a], pts[b]
                    ax.plot([xa, xb], [ya, yb], "-", linewidth=1.2)
            xs = [p[0] for p in pts if p is not None]
            ys = [p[1] for p in pts if p is not None]
            ax.scatter(xs, ys, s=10)

    # hide unused axes
    for k in range(n, rows * cols):
        axes[k // cols][k % cols].axis("off")
    plt.tight_layout()
    plt.show()

show_all_cams(window._current_frame)

## 8. Track & identity colors

Color is **derived** for tracks (no stored map — it's a pure function of `track_idx`)
and **stored** for identities (on `Identity.color`).

```
TRACK_COLORS                       # list[str], 20 hex strings (Green-Armytage palette)
get_track_color(track_idx)         # → hex; "#888888" for None / unlinked
get_identity_color(id, fallback)   # → hex; falls back to track color if id is None
next_palette_color(n_existing)     # default color for the next new Identity

Identity.id      / .name / .color  # color persists across sessions
session.color_mode                 # "track" | "identity"
```

**Rule:** if `session.color_mode == "identity"` and the (frame, cam, track) maps
to an identity, use `identity.color`; otherwise use `get_track_color(track_idx)`.

In [ ]:
from colors import TRACK_COLORS, get_track_color, get_identity_color

# Track idx/name -> hex (always defined; cycles past 20)
track_idx_to_color  = {i: get_track_color(i)  for i in range(len(session.tracks))}
track_name_to_color = {n: get_track_color(i)
                       for i, n in enumerate(session.tracks)}

# Identity id/name -> hex (only for identities that actually exist)
identity_id_to_color   = {ident.id:   ident.color for ident in session.identities}
identity_name_to_color = {ident.name: ident.color for ident in session.identities}

print("palette size:", len(TRACK_COLORS))
print("track_idx -> color:", track_idx_to_color)
print("track_name -> color:", track_name_to_color)
print("identity_id -> color:", identity_id_to_color)
print("current color_mode:", session.color_mode)

In [ ]:
def overlay_color_for(session, inst, frame_idx, cam):
    """Exactly what the on-screen overlay would paint for this instance right now.

    Mirrors overlay_renderer.draw_overlay_for_camera:
      - identity mode: identity.color, else "#888888"
      - track    mode: TRACK_COLORS[track_idx % 20], or "#888888" if untracked
    """
    if session.color_mode == "identity":
        ident_id = session.get_identity_id_for_track(frame_idx, cam, inst.track_idx)
        ident = session.get_identity(ident_id)
        return ident.color if ident else "#888888"
    return get_track_color(inst.track_idx)

fg = session.frame_group(window._current_frame)
if fg is not None:
    for cam, insts in fg.instances.items():
        for inst in insts:
            c = overlay_color_for(session, inst, window._current_frame, cam)
            print(f"{cam}  track_idx={inst.track_idx}  ->  {c}")
        break  # just the first cam for brevity

In [ ]:
# Visual swatch: full track palette + each existing Identity
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(12, 2.4))
ax.set_xlim(0, max(len(TRACK_COLORS), len(session.tracks)))
ax.set_ylim(0, 3)
ax.axis("off")

# Row 0: full palette
for i, c in enumerate(TRACK_COLORS):
    ax.add_patch(Rectangle((i, 0), 1, 1, facecolor=c, edgecolor="#222"))
    ax.text(i + 0.5, 0.5, str(i), ha="center", va="center", fontsize=8)
ax.text(-0.4, 0.5, "palette", ha="right", va="center")

# Row 1: per-track colors used in this session
for i, name in enumerate(session.tracks):
    ax.add_patch(Rectangle((i, 1.1), 1, 0.8, facecolor=get_track_color(i), edgecolor="#222"))
    ax.text(i + 0.5, 1.5, name, ha="center", va="center", fontsize=8)
ax.text(-0.4, 1.5, "tracks", ha="right", va="center")

# Row 2: identities (if any)
for i, ident in enumerate(session.identities):
    ax.add_patch(Rectangle((i, 2.1), 1, 0.8, facecolor=ident.color, edgecolor="#222"))
    ax.text(i + 0.5, 2.5, ident.name, ha="center", va="center", fontsize=8)
ax.text(-0.4, 2.5, "identities", ha="right", va="center")

plt.tight_layout(); plt.show()

## 9. Calibration (intrinsics + extrinsics)

Per-camera OpenCV-style calibration lives on `session.cameras: list[Camera]`.

```
Camera.name      str                       # e.g. "back", "top"
Camera.matrix    list[list[float]] (3x3)   # intrinsic K
Camera.dist      list[float] (≥4)          # OpenCV distortion [k1, k2, p1, p2, k3, ...]
Camera.rvec      list[float] (3)           # Rodrigues rotation (world -> camera)
Camera.tvec      list[float] (3)           # translation (world -> camera)
Camera.size      tuple[int, int]           # (width, height) px

session.has_calibration   # False -> the values above are identity-K stubs
```

Convention is the same as `cv2.solvePnP` / `cv2.projectPoints` — `rvec`, `tvec` take a world point to camera space. The on-disk file can be `calibration.toml` (preferred) or `calibration.json`; the parser also accepts the aliases `K`, `dist`, `rotation`/`rvec`, `translation`/`tvec`.

In [ ]:
import numpy as np

print("has_calibration:", session.has_calibration)

cam_by_name = {c.name: c for c in session.cameras}

def cam_arrays(cam):
    """-> (K (3,3), dist (k,), rvec (3,), tvec (3,)) all float64 numpy."""
    K    = np.asarray(cam.matrix, dtype=float)
    dist = np.asarray(cam.dist,   dtype=float).ravel()
    rvec = np.asarray(cam.rvec,   dtype=float).ravel()
    tvec = np.asarray(cam.tvec,   dtype=float).ravel()
    return K, dist, rvec, tvec

first = session.cameras[0]
K, dist, rvec, tvec = cam_arrays(first)
print(f"\n{first.name}")
print("  size =", first.size)
print("  K =\n", K)
print("  dist =", dist)
print("  rvec =", rvec)
print("  tvec =", tvec)

In [ ]:
# Tabular summary of every camera's calibration
import pandas as pd

pd.DataFrame([{
    "name": c.name,
    "width":  c.size[0],
    "height": c.size[1],
    "fx": c.matrix[0][0], "fy": c.matrix[1][1],
    "cx": c.matrix[0][2], "cy": c.matrix[1][2],
    "k1": c.dist[0] if len(c.dist) > 0 else 0.0,
    "rvec": [round(v, 4) for v in c.rvec],
    "tvec": [round(v, 2) for v in c.tvec],
} for c in session.cameras])

In [ ]:
# Pure-numpy Rodrigues (no OpenCV dependency) and 4x4 extrinsic.
import numpy as np

def rodrigues(rvec: np.ndarray) -> np.ndarray:
    """Rotation vector (3,) -> rotation matrix (3,3). Matches cv2.Rodrigues."""
    rvec = np.asarray(rvec, dtype=float).reshape(3)
    theta = float(np.linalg.norm(rvec))
    if theta < 1e-12:
        return np.eye(3)
    k = rvec / theta
    K = np.array([[0, -k[2], k[1]],
                  [k[2], 0, -k[0]],
                  [-k[1], k[0], 0]])
    return np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * (K @ K)

def extrinsic_4x4(cam) -> np.ndarray:
    """World -> camera 4x4 homogeneous transform."""
    K, _, rvec, tvec = cam_arrays(cam)
    R = rodrigues(rvec)
    Rt = np.eye(4)
    Rt[:3, :3] = R
    Rt[:3,  3] = tvec
    return Rt

for c in session.cameras[:3]:
    Rt = extrinsic_4x4(c)
    # Camera center in world coords: C = -R^T t
    R = Rt[:3, :3]; t = Rt[:3, 3]
    C_world = -R.T @ t
    print(f"{c.name:>6s}  camera center in world = {np.round(C_world, 2)}")

In [ ]:
# Project an arbitrary world point into every camera (with distortion).
# Requires opencv-python: `uv add opencv-python` if not already installed.
import numpy as np

def project_with_distortion(cam, X_world: np.ndarray) -> np.ndarray:
    """X_world (N,3) -> uv (N,2) using OpenCV's full plumb-bob model."""
    import cv2  # noqa: WPS433 — keep optional
    K, dist, rvec, tvec = cam_arrays(cam)
    uv, _ = cv2.projectPoints(X_world.reshape(-1, 1, 3), rvec, tvec, K, dist)
    return uv.reshape(-1, 2)

# Example: project the world origin into every camera
try:
    import cv2  # noqa: F401
    X = np.array([[0.0, 0.0, 0.0]])
    for c in session.cameras:
        uv = project_with_distortion(c, X)
        print(f"{c.name:>6s}  world (0,0,0) -> uv = {np.round(uv[0], 1)}")
except ImportError:
    print("opencv-python not installed; skipping projection example.")
    print("Install with: uv add opencv-python")

In [ ]:
# Path to the on-disk calibration file (handy for re-export / reload)
from calibration import find_calibration
calib_path = find_calibration(session.folder)
print("calibration source:", calib_path)